## RAG


In [2]:
import chromadb
from pypdf import PdfReader
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
import os


In [3]:
# The .env lives at the repo root, the database and index next to this script
ROOT = Path.cwd().parent
PDF_PATH = ROOT / "data" / "rag_docs" / "irs_pub_523.pdf"

In [4]:
load_dotenv(ROOT / ".env")
BASE_URL = "https://openrouter.ai/api/v1"


In [5]:
client = OpenAI(base_url=BASE_URL)

# 1. Read PDF
reader = PdfReader(PDF_PATH)


In [6]:

text = ""

for page in reader.pages:
    text += page.extract_text() or ""

# 2. Split text into chunks
chunk_size = 500

chunks = [
    text[i:i + chunk_size]
    for i in range(0, len(text), chunk_size)
]

# 3. Create ChromaDB
chroma = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma.get_or_create_collection(
    name="pdf_documents"
)

# 4. Generate embeddings
for i, chunk in enumerate(chunks):

    embedding = client.embeddings.create(
        model="google/gemini-embedding-001",
        input=chunk
    ).data[0].embedding

    collection.upsert(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[embedding]
    )

print("PDF stored in ChromaDB!")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FirstChar': 65, '/FontDescriptor': IndirectObject(3640, 0, 2573584491456), '/BaseFont': '/KHAMKK+Helvetica-Black', '/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/LastChar': 85, '/Widths': [778, 0, 778, 0, 0, 0, 0, 0, 389, 0, 0, 0, 0, 833, 833, 0, 0, 0, 0, 722, 833]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FirstChar': 33, '/FontDescriptor': IndirectObject(3641, 0, 2573584491456), '/BaseFont': '/KHAMKL+Helvetica-Bold', '/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/LastChar': 33, '/Widths': [333]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FirstCh

KeyboardInterrupt: 

In [7]:
# User question
question = input("Ask a question: ")

# 1. Convert question into embedding
question_embedding = client.embeddings.create(
    model="google/gemini-embedding-001",
    input=question
).data[0].embedding

# 2. Search ChromaDB
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)


In [16]:
print(results["documents"][0])

['ou \nown and live in just one home, then that property is your \nmain home. If you own or live in more than one home, then \nyou must apply a “facts and circumstances” test to deter-\nmine which property is your main home. While the most \nimportant factor is where you spend the most time, other \nfactors are relevant as well. They are listed below. The \nmore of these factors that are true of a home, the more \nlikely that it is your main home. \n• The address listed on your:\n1. U.S. Postal Service addr', 'rk \nlocation is 65 miles from the home.\n• You had no previous work location and you began a \nnew job at least 50 miles from the home.\n• Either of the above is true of your spouse, a co-owner \nof the home, or anyone else for whom the home was \ntheir residence.\n6 Publication 523 (2025)Page 7 of 27  Fileid: … cation-523/2025/a/xml/cycle03/source 7:18 - 27-Feb-2026\nThe type and rule above prints on all proofs including departmental reproduction proofs. MUST be removed before 

In [12]:

# 3. Get relevant documents
context = "\n".join(
    results["documents"][0]
)

# 4. Send context to OpenAI
response = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": "Answer using only the provided context."
        },
        {
            "role": "user",
            "content": f"""
            Context:
            {context}

            Question:
            {question}
            """
        }
    ]
)

print("\nAI Answer:")
print(response.choices[0].message.content)


AI Answer:
This document does not provide information or instructions on how to change your address. It only lists "The address listed on your: U.S. Postal Service address, Voter Registration Card, Federal and state tax returns, and Driver’s license or car registration" as factors in determining your main home.
